In [1]:
!pip install PyMuPDF
!pip install openai 
!pip install dotenv 
!pip install markdown-pdf

  Using cached pymupdf-1.26.7-cp310-abi3-macosx_11_0_arm64.whl.metadata (3.4 kB)
Using cached pymupdf-1.26.7-cp310-abi3-macosx_11_0_arm64.whl (22.5 MB)
  Using cached openai-2.14.0-py3-none-any.whl.metadata (29 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached jiter-0.12.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (5.2 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.41.5-cp312-cp312-macosx_11_0_arm64.whl.metadata (7.3 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
Using cached openai-2.14.0-py3-none-any.whl (1.1 MB)
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
Using cached jiter-0.12.0-cp312-cp312-macosx_11_0_arm64.whl (319 kB)
Using cached pydantic-2.12.5-py3-none-any.whl 

In [13]:
import requests
from bs4 import BeautifulSoup

def get_jd_description(job_url):
    try:
        response = requests.get(job_url)
        soup = BeautifulSoup(response.content, 'html.parser')

        for script in soup(["script", "style", "nav", "footer"]):
            script.decompose()
        text = soup.get_text(separator = " ")
        return " ".join(text.split())
    except Exception as e:
        return f"Exception occured: {str(e)}"

job_url = "https://job-boards.greenhouse.io/greenhouse/jobs/7316197"
jd_text = get_jd_description(job_url)

## Read the CV 

In [15]:
# PyMuPDF
import fitz 

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

resume_text = extract_text_from_pdf("master_cv.pdf")

## Taylor the CV

In [21]:
from openai import OpenAI
import os 
from dotenv import load_dotenv

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

In [22]:
client = OpenAI()

In [26]:
def taylor_resume(client, resume_text, jd_text):
    prompt= f"""
    You are an expert Resume Writer and Applicant Tracking System Optimization Specialist.
    JOB DESCRIPTION (JD):
    {jd_text}
    MY MASTER RESUME:
    {resume_text}
    TASK:
    Rewrite my resume to perfectly match the Job Description.
    RULES:
    1. Use the EXACT keywords from the JD in skills and summary sections.
    2. Rephrase bullet points to match the responsibilities and metrics they care about.
    3. Do NOT lie or invent new experience.
    4. Only use real experience from my resume.
    5. Output in clean, well-formatted MARKDOWN.
    """
    response = client.responses.create(
        model = "gpt-5-nano",
        input = prompt
    )
    return response

In [27]:
response = taylor_resume(client, resume_text, jd_text)
markdown_text = response.output_text

In [28]:
print(markdown_text)

SHRINIVASAN SANKAR
3 Holsey Lane, Bletchley, MK2 3FH, United Kingdom
+44 7490 757555 | vasan.shrini@gmail.com | https://www.ai-bites.net

Summary
- Senior ML Ops Engineer with 8+ years of AI experience, operating within an Applied Machine Learning team to own the infrastructure and processes that bring machine learning models to life.
- I own the full model lifecycle from prototypes to production-ready systems, and I create, manage, and improve continuous integration and delivery pipelines to support rapid iteration and deployment.
- I advocate for observability practices like monitoring, logging, and alerting, and I focus on maintaining high data quality and performance in production.
- I collaborate across ML engineers, data scientists, and software teams to deliver impactful solutions and production-grade ML services.

Skills (selected keywords aligned with the job description)
- MLOps, DevOps
- continuous integration and delivery pipelines
- Observability practices like monitoring,

## Save the modified CV as PDF

In [29]:
from markdown_pdf import MarkdownPdf, Section

def save_as_pdf(markdown_content, output_file = "tailored_cv.pdf"):
    pdf = MarkdownPdf(toc_level=0)

    css_style="""
    body { font-family: 'Helvetica', sans-serif; font-size: 12px; line-height: 1.4; color: #333; }
    h1 { font-size: 24px; border-bottom: 2px solid #333; margin-bottom: 10px; text-transform: uppercase; }
    h2 { font-size: 16px; border-bottom: 1px solid #ccc; margin-top: 20px; text-transform: uppercase; color: #555; }
    ul { padding-left: 20px; }
    li { margin-bottom: 5px; }
    strong { color: #000; }
    """
    
    pdf.add_section(Section(markdown_content), user_css = css_style)
    pdf.save(output_file)

In [30]:
save_as_pdf(markdown_text)

## Rate the match

In [31]:
def rate_match(job_description, resume_text):
    prompt = f"""
    You are an expert resume writer and Applicant  Tracking System specialist.
    You re given a job description and a resume below.
    Job Description:
    {job_description}
    Resume: 
    {resume_text}
    TASK:
    Compare the key words in job description to the key words in the resume.
    Come up with a matching score between 1 and 100. 1 is the least match and 100 is the most match.
    RULES:
    1. Do NOT come up with a score below 1 or above 100.
    2. Do NOT give any other output other than just a single score.
    """
    response = client.responses.create(
        model = "gpt-5-nano",
        input = prompt
    )
    return response

In [32]:
match_response = rate_match(jd_text, resume_text)
print(f"Matching score is: {match_response.output_text}")

Matching score is: 46


In [33]:
match_response = rate_match(jd_text, markdown_text)
print(f"Matching score is: {match_response.output_text}")

Matching score is: 72
